# SmolLM2-135M Training (Unsloth + LoRA)

This notebook trains a student assignment metadata extractor and exports GGUF for Ollama.

## Generate Synthetic Data

In [ ]:
import json
import random
from pathlib import Path

# Configuration
DATASET_SIZE = 1250
OUTPUT_FILE = "data/dataset.json"

# Constants & Variations
STUDENT_NUMBER_KEYS = [
    "Student No", "Stu. ID", "Student ID", "ID",
    "Reg No", "Registration No", "Index No", "Reg. Number",
    "Stuednt No", "Stdnt ID", "Student #" # Added typos and symbols
]
STUDENT_NAME_KEYS = [
    "Name", "Full Name", "Student Name", "Student", "Stu. Name",
    "Nmae", "Full Nme", "Candidate Name"
]
ASSIGNMENT_KEYS = [
    "Assignment #", "Assignment No", "HW", "Task No",
    "Submission No", "Assgn #", "Worksheet No",
    "Assignemnt #", "HW No.", "Practical"
]

# Base separators, will be modified dynamically with chaotic spacing
BASE_SEPARATORS = [":", "-", "=", ""]
LINE_BREAKS = ["\n", " | ", ", ", "  ", "\n\n", " \t "]

NAMES = [
    "Amal Perera", "Nimal Silva", "Kasun Fernando", "Dilini Rathnayake",
    "Chamara Bandara", "Sithum Jayawardena", "Amali Gunasekara",
    "Priyanka De Silva", "Ruwan Kumara", "Tharushi Peiris",
    "M.N. Perera", "Silva, Nimal", "Kasun A. Fernando"
]

NOISE_TEXTS = [
    "Course: Computer Science 101",
    "University of Technology",
    "Final Submission",
    "Please grade fairly!",
    "Module: Data Structures",
    "Due Date: 12th October",
    "Word count: 1200",
    "Honor Code Pledged",
    "---START OF DOCUMENT---"
]

# Mutation Helpers
def mutate_name(name: str) -> str:
    """Randomly alters name capitalization or spacing."""
    chance = random.random()
    if chance < 0.15: return name.upper()
    if chance < 0.30: return name.lower()
    if chance < 0.40: return f"Mr. {name}" if random.random() < 0.5 else f"Ms. {name}"
    if chance < 0.50: return "  ".join(name.split()) # Weird double spacing
    return name

def mutate_student_num(num: int) -> str:
    """Randomly adds prefixes or formats to the raw integer."""
    num_str = str(num)
    chance = random.random()
    if chance < 0.2: return f"CS-{num_str}"
    if chance < 0.4: return f"S{num_str}"
    if chance < 0.5: return f"{num_str[:4]}-{num_str[4:]}" # e.g., 2021-0045
    return num_str

def mutate_assign_num(num: int) -> str:
    """Randomly pads or modifies the assignment number."""
    chance = random.random()
    if chance < 0.2: return f"0{num}" # Zero padding (e.g., "03")
    if chance < 0.4: return f"{num}A" # Sub-assignments
    if chance < 0.5: return f"Part {num}"
    return str(num)

def chaotic_separator(sep: str) -> str:
    """Adds irregular spacing or markdown around a separator."""
    left_space = " " * random.randint(0, 2)
    right_space = " " * random.randint(0, 2)
    return f"{left_space}{sep}{right_space}"

def apply_markdown(text: str) -> str:
    """Occasionally wraps text in markdown bold/italic tags."""
    if random.random() < 0.15: return f"**{text}**"
    if random.random() < 0.25: return f"_{text}_"
    return text

# Functions
def make_example(student_num_raw: int, name_raw: str, assign_num_raw: int) -> dict:
    # 1. Mutate the actual values to be messy
    student_num = mutate_student_num(student_num_raw)
    name = mutate_name(name_raw)
    assign_num = mutate_assign_num(assign_num_raw)

    # 2. Pick keys and apply occasional markdown (e.g., **Name:**)
    sk = apply_markdown(random.choice(STUDENT_NUMBER_KEYS))
    nk = apply_markdown(random.choice(STUDENT_NAME_KEYS))
    ak = apply_markdown(random.choice(ASSIGNMENT_KEYS))

    # 3. Apply chaotic spacing to separators
    sep_s = chaotic_separator(random.choice(BASE_SEPARATORS))
    sep_n = chaotic_separator(random.choice(BASE_SEPARATORS))
    sep_a = chaotic_separator(random.choice(BASE_SEPARATORS))

    # Create individual field strings
    parts = [
        f"{sk}{sep_s}{student_num}",
        f"{nk}{sep_n}{name}",
        f"{ak}{sep_a}{assign_num}"
    ]

    # 30% chance to shuffle the order of the fields
    if random.random() < 0.3:
        random.shuffle(parts)

    lb = random.choice(LINE_BREAKS)
    text = lb.join(parts)

    # 40% chance to add surrounding noise
    if random.random() < 0.4:
        noise = random.choice(NOISE_TEXTS)
        if random.random() < 0.5:
            text = f"{noise}\n{text}"
        else:
            text = f"{text}\n{noise}"

    return {
        "instruction": "Extract student info as JSON from the following text.",
        "input": text,
        # The output JSON MUST use the clean, unmutated strings to train the model to normalize data
        "output": json.dumps({
            "student_number": str(student_num_raw),
            "student_name": name_raw,
            "assignment_number": str(assign_num_raw),
        })
    }

def generate_dataset(size: int) -> list[dict]:
    dataset = [
        make_example(20210000 + i, random.choice(NAMES), (i % 10) + 1) for i in range(size)
    ]
    random.shuffle(dataset)
    return dataset

# --- Execution ---
if __name__ == "__main__":
    output_path = Path(OUTPUT_FILE)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Generate and save the dataset
    dataset = generate_dataset(DATASET_SIZE)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(dataset, f, indent=2)

    print(f"Generated {len(dataset)} highly-varied examples and saved to: {output_path}")

    # Print a preview of a messy example
    print("\n--- Preview of a Generated Example ---")
    print(json.dumps(dataset[0], indent=2))

In [ ]:
!pip install unsloth trl datasets transformers accelerate

In [69]:
from unsloth import FastLanguageModel

import torch
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

import json
from pathlib import Path

In [70]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"
MAX_SEQ_LENGTH = 512
DATASET_PATH = Path("data/dataset.json")
HF_OUTPUT_DIR = "./smollm-student-extractor"
GGUF_OUTPUT_DIR = "smollm-student-gguf"

if not DATASET_PATH.exists():
    raise FileNotFoundError("data/dataset.json not found. Run: uv run python data/generate_dataset.py")

In [ ]:
with DATASET_PATH.open(encoding="utf-8") as f:
    raw = json.load(f)

if not raw:
    raise ValueError("Dataset is empty. Generate examples first.")

def format_example(ex):
    return {
        "text": (
            f"### Instruction:\n{ex['instruction']}\n\n"
            f"### Input:\n{ex['input']}\n\n"
            f"### Response:\n{ex['output']}"
        )
    }

dataset = Dataset.from_list(raw).map(format_example, remove_columns=list(raw[0].keys()))
dataset

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

In [ ]:
cuda_available = torch.cuda.is_available()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=4,
        learning_rate=2e-4,
        fp16=cuda_available and not torch.cuda.is_bf16_supported(),
        bf16=cuda_available and torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir="./outputs",
        save_strategy="epoch",
        report_to="none",
    ),
)

trainer.train()

In [74]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# --- Save to Hugging Face directly ---
HF_REPO = "nimendraai/SmolLM2-360M-Assignment-Metadata-Extractor"

print("Pushing standard weights to Hub...")
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print("Pushing GGUF to Hub...")
model.push_to_hub_gguf(
    HF_REPO,
    tokenizer,
    quantization_method="q4_k_m",
)
print("Done! All formats uploaded.")

In [ ]:
model.save_pretrained(HF_OUTPUT_DIR)
tokenizer.save_pretrained(HF_OUTPUT_DIR)

model.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

print("Done — GGUF ready for Ollama.")